# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tessa-Saumu/FlyRank-ML-Internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

> **Freestyle submission, V2 edition.** This notebook belongs to my final, freestyle capstone: *which ranking pages are losing clicks to AI-answer search before it shows, and which should an editor protect first?* The pipeline was rebuilt as **V2**: the whole process was written down and hashed in `work/V2_SPEC.md` before any V2 code ran. V1 stays in the repository as the first process, and every V1 number shown here is labelled V1. My earlier Lane 2 (refresh) experiments are kept for the record in `work/archive/` and are not part of this submission.

**What this card asks:** task type, target or proxy, success metric, the unit of analysis as a real dataframe, and why
ML beats a fixed rule. **The principle it serves:** decision first. Name the metric before training, and make the
target an observed outcome, never a rule's output.

In [1]:
# Setup: works locally and in Colab. Reads committed receipts; the warehouse is touched only by the opt-in rerun.
import importlib.util, json, os, sys
if any(importlib.util.find_spec(p) is None for p in ["duckdb", "sklearn", "scipy", "matplotlib"]):
    get_ipython().run_line_magic("pip", "install -q duckdb huggingface_hub pandas scikit-learn scipy matplotlib")
here = os.path.abspath(os.getcwd())
ROOT = next((p for p in [here, os.path.dirname(here), os.path.dirname(os.path.dirname(here))]
             if os.path.isdir(os.path.join(p, "work", "scripts"))), None)
if ROOT is None:  # fresh Colab runtime: fetch the code (no data lives in git)
    get_ipython().system("git clone -q https://github.com/Tessa-Saumu/FlyRank-ML-Internship.git")
    ROOT = os.path.abspath("FlyRank-ML-Internship")
sys.path.insert(0, os.path.join(ROOT, "work", "scripts"))
OUT, FIG = os.path.join(ROOT, "work", "outputs"), os.path.join(ROOT, "work", "figures")
import numpy as np, pandas as pd
import warnings; warnings.filterwarnings("ignore")
from IPython.display import Image, display, Markdown
import wh
RERUN_PIPELINE = False   # opt-in: True recomputes from the pinned snapshot (needs HF_TOKEN). V2 steps then run in a scratch copy, never in the repo.
DATA = wh.data_available(try_download=RERUN_PIPELINE)
print("pinned warehouse snapshot available:", DATA, "| revision", wh.REV[:8], "| RERUN_PIPELINE =", RERUN_PIPELINE)
receipt = lambda name: json.load(open(os.path.join(OUT, name)))
pd.set_option("display.width", 170); pd.set_option("display.max_columns", 30)

# V2 receipts (aggregates only) and the helpers every notebook shares.
v2 = lambda n: receipt(f"v2_{n}.json")
BASE = ["rule_position_slipping", "mean_reversion_ctr_excess", "client_shock", "persistence_prev_ratio"]
BASE_LABEL = {"rule_position_slipping": "editor's rule", "mean_reversion_ctr_excess": "CTR above peers",
              "client_shock": "client shock", "persistence_prev_ratio": "persistence"}
def v2_months():
    """(month, role, evaluation block) for the four evaluated months. The logistic champion is in every block."""
    vf, cf, ts = v2("validation_folds"), v2("confirmation_apr"), v2("test")
    return [("2026-02", "validation", vf["folds"]["validation_1"]), ("2026-03", "validation", vf["folds"]["validation_2"]),
            ("2026-04", "confirmation", cf["evaluation"]), ("2026-05", "test", ts["evaluation"])]
def champ(ev):
    return ev["models"].get("LOGISTIC") or ev["models"]["SELECTED:LOGISTIC"]
def fold_table():
    """One row per evaluated month: the champion against the four frozen baselines, same rows, same metric."""
    rows = []
    for m, role, ev in v2_months():
        lg, best = champ(ev), ev["best_baseline_by_ap"]
        rows.append({"month": m, "role": role, "base rate": ev["base_rate"], "rule P@50": ev["models"][BASE[0]]["p@50"],
                     "best baseline (by AP)": BASE_LABEL[best], "best baseline AP": ev["models"][best]["ap"],
                     "V2 AP": lg["ap"], "V2 AP 95% CI": lg["ap_ci"], "V2 P@50": lg["p@50"], "V2 P@50 95% CI": lg["p@50_ci"],
                     "P@50, unmeasurable = miss": lg["p@50_unmeasurable_as_miss"],
                     "rule P@50, unmeasurable = miss": ev["models"][BASE[0]]["p@50_unmeasurable_as_miss"],
                     "rule share of click shortfall (top 50)": ev["models"][BASE[0]]["clicks_lost_captured"],
                     "V2 share of click shortfall (top 50)": lg["clicks_lost_captured"]})
    return pd.DataFrame(rows)

pinned warehouse snapshot available: True | revision 50cbf7c3 | RERUN_PIPELINE = False


## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Question:** which kind of ML task is this? **Why it matters (FlyRank principle: decision first):** the decision is
"which 50 pages first", so the task is a ranking, not a yes or no for every page.

| Item | Choice | Why |
|---|---|---|
| Task type | supervised ranking: binary label, pages ranked by predicted probability | the decision is "which 50 first", not "yes or no for every page" |
| Comparison | four frozen baselines, and four learned candidates compared under one rule fixed in advance (ML-08) | the model must beat simple rules, not just chance |

In [2]:
summ = receipt("erosion_frame_summary.json")
spec = v2("spec")
print("task: supervised ranking | review budget K =", spec["metrics"]["headline_k"], "| candidates:", spec["candidates"], "| baselines:", len(spec["baselines"]))

task: supervised ranking | review budget K = 50 | candidates: ['LOGISTIC', 'HGB_CLF', 'POISSON_RATE', 'ENSEMBLE'] | baselines: 4


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Question:** what is the target, and where does the label come from? **Why it matters (FlyRank principle: an observed
outcome, never a rule's output):** a label defined by a rule can be recovered by that rule, which is what ended my Lane 2
study.

| Item | Choice | Why |
|---|---|---|
| Target | `erode = 1` if clicks in months *o* and *o+1* < 0.75 x expected clicks | observed in a later window; nobody's rule defines it |
| Expected clicks | outcome impressions x current CTR (month *o-1*, shrunk) x position adjustment | compares the page with itself at its own rank |

The question, the unit, the eligibility rule and this label are unchanged from V1. V2 changes how features, models and the
final model are chosen, not what is predicted.

**The timeline (what is known when).**

```
decision on day 1 of month o
features : months o-2 and o-1 (plus o-3 for one trend feature)   -> closed before the decision
label    : months o and o+1                                         -> observed after the decision
training : only origins <= o-2, whose labels have matured by month o-1
```

**The V2 split, one row per role.** The outcome windows of training, validation and test never overlap in the way that
matters: every training label closes before the scored month begins, and the two validation windows end before the test
window opens.

| Role | Scored origin | Outcome window | Trained on origins | Used for decisions |
|---|---|---|---|---|
| Validation fold 1 | 2026-02 | Feb to Mar | 2025-12 | yes |
| Validation fold 2 | 2026-03 | Mar to Apr | 2025-12, 2026-01 | yes |
| Confirmation fold | 2026-04 | Apr to May | 2025-12 to 2026-02 | no, reported only |
| Test (sealed) | 2026-05 | May to Jun | 2025-12 to 2026-03 (refit) | no, scored once |
| Live | 2026-07 | none yet | 2025-12 to 2026-05 | queue only |

The confirmation fold's outcome window (Apr to May) overlaps the test window, which is exactly why it cannot drive any
choice: it is computed after the selection is written to disk and cannot change it.

**Three guards that make the label honest.**

| Guard | What it does | Why it is needed |
|---|---|---|
| Shrinkage (empirical Bayes, m = 1,000 impressions) | pulls month *o-1* CTR toward the client's position-adjusted CTR | at about 0.4% CTR, a page with 20 clicks has a noisy rate; shrinkage stops noise from posing as erosion |
| Fixed position curve (Oct-Nov 2025 only) | adjusts the expectation if rank moves | a page that drops rank loses clicks for a known reason; that is not AI-answer erosion |
| Poisson noise floor | simulates the label on pure sampling noise | shows how much of the base rate chance alone would produce |

The label anchors on the **latest** month, shrunk toward the **client**, not toward the page's own history. Two earlier
versions leaked momentum into the label. ML-05 (`w03_feature_leakage_check`) shows how I caught them.

In [3]:
o = pd.DataFrame(summ["origins"]).dropna(subset=["base_rate"])
o["signal_over_noise"] = (o.base_rate / o.null_rate).round(1)
display(o[["origin", "measurable", "clients", "base_rate", "null_rate", "signal_over_noise"]])
tl = v2("leakage_checks")["timeline"]
display(pd.DataFrame([{"role": k, "scored origin": v["score_origin"], "trained on": ", ".join(v["train_outcome_windows_end"]),
                       "last training outcome day": max(v["train_outcome_windows_end"].values()), "origin first day": v["origin_first_day"],
                       "training ends before origin": v["latest_training_outcome_end_before_origin"]} for k, v in tl.items()]))

,origin,measurable,clients,base_rate,null_rate,signal_over_noise
0,2025-12,5378.0,19,0.3241,0.0515,6.3
1,2026-01,5785.0,19,0.2430,0.0455,5.3
2,2026-02,7083.0,21,0.2273,0.0517,4.4
3,2026-03,8613.0,29,0.3382,0.0513,6.6
4,2026-04,11186.0,30,0.3079,0.0580,5.3


,role,scored origin,trained on,last training outcome day,origin first day,training ends before origin
0,validation_1,2026-02,2025-12,2026-01-31,2026-02-01,True
1,validation_2,2026-03,"2025-12, 2026-01",2026-02-28,2026-03-01,True
2,confirmation,2026-04,"2025-12, 2026-01, 2026-02",2026-03-31,2026-04-01,True
3,test,2026-05,"2025-12, 2026-01, 2026-02, 2026-03",2026-04-30,2026-05-01,True
4,live,2026-07,"2025-12, 2026-01, 2026-02, 2026-03, 2026-04, 2...",2026-06-30,2026-07-01,True


**What it means.** Pure noise would label about 5% of pages as eroding (0.045 to 0.071 across the training, validation, confirmation and test months). The observed
rate is 0.23 to 0.34 in the four evaluated months, several times higher, so most positives are real changes in click
share at a stable rank, not counting noise. The second table is the split above, read from the V2 leakage receipt: in every
row the last training outcome day is before the origin's first day.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Question:** which one number means "good"? **Why it matters (FlyRank principle: name the metric before training):** a
metric chosen after seeing results can be bent to flatter the model.

| Item | Choice | Why |
|---|---|---|
| Selection criterion | average precision (AP) over the two validation folds, fixed in the spec before any run | one pre-declared number, so the winner is not picked by whichever metric flatters it |
| Headline, reported next to the base rate | precision@50, with client-bootstrap 95% intervals | 50 is the monthly review budget; P@50 is the operational metric |
| Also reported, never used to choose | P@20, P@100, AUC, lift, within-client P@10, clicks-lost captured, top-client share of the top 50 | P@50 alone has wide intervals (about +/-0.2 at K = 50) |
| Conservative variant | P@50 counting unmeasurable pages as misses | pages that lose almost all visibility cannot be labelled; counting them as misses is the cautious bound |

In [4]:
b = pd.DataFrame([{"month": m, "role": r, "base rate": ev["base_rate"], "noise-only rate": ev["noise_only_rate"],
                   "eligible pages": ev["n_eligible"], "measurable": ev["n_measurable"], "lost-visibility share": ev["lost_visibility_share"],
                   "clients": ev["n_clients"], "effective clients": ev["effective_clients"]} for m, r, ev in v2_months()])
print("Every metric below is read next to the month's base rate, and with the effective number of clients:"); display(b)

Every metric below is read next to the month's base rate, and with the effective number of clients:


,month,role,base rate,noise-only rate,eligible pages,measurable,lost-visibility share,clients,effective clients
0,2026-02,validation,0.2273,0.0517,7630,7083,0.0717,20,4.78
1,2026-03,validation,0.3382,0.0513,8984,8613,0.0413,27,4.42
2,2026-04,confirmation,0.3079,0.0580,12319,11186,0.0920,28,5.36
3,2026-05,test,0.2397,0.0709,13082,10977,0.1609,30,5.80


**What it means.** The effective number of clients (1 divided by the sum of squared row shares) is 4.42 to 5.80 in the
four evaluated months, although there are 20 to 30 clients. The top three clients hold 61% to
74% of the rows. So every interval is built by resampling whole clients, and the
effective sample size is a limit on what can be claimed, not a detail.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**Question:** one row is one what? **Why it matters (FlyRank principle: one explicit grain):** a reviewer should see
exactly what one row is. IDs are pseudonymous and used only for grouping, so they are not shown. The frame is built by
`work/scripts/erosion_frame.py` from the cached page-month table.

In [5]:
summ = receipt("erosion_frame_summary.json")
display(pd.DataFrame(summ["origins"]))
if DATA and os.path.exists(wh.CACHE + "/erosion_frame.parquet"):
    import pyarrow.parquet as pq   # read only the validation month: the sealed test month is never loaded here
    cols = ["origin", "imp_f", "clk_f", "pos_f", "pos_m1", "ctr_now", "imp_out", "clk_out", "pos_out", "exp_out", "erode"]
    f = pq.read_table(wh.CACHE + "/erosion_frame.parquet", columns=cols, filters=[("origin", "==", "2026-03")]).to_pandas()
    display(f.sample(6, random_state=1).round(4)); print("rows in the 2026-03 slice:", len(f))

,origin,eligible,measurable,clients,base_rate,null_rate,lost_visibility,sealed,live_no_outcome
0,2025-12,5905,5378.0,19,0.3241,0.0515,0.0892,False,False
1,2026-01,6642,5785.0,19,0.2430,0.0455,0.1290,False,False
2,2026-02,7630,7083.0,21,0.2273,0.0517,0.0717,False,False
3,2026-03,8984,8613.0,29,0.3382,0.0513,0.0413,False,False
4,2026-04,12319,11186.0,30,0.3079,0.0580,0.0920,False,False
5,2026-05,13082,NaN,33,NaN,NaN,NaN,True,False
6,2026-07,14100,NaN,37,NaN,NaN,NaN,False,True


,origin,imp_f,clk_f,pos_f,pos_m1,ctr_now,imp_out,clk_out,pos_out,exp_out,erode
3011,2026-03,8126.0,20.0,5.4100,5.4100,0.0027,40196.0,117.0,4.2338,113.3047,0.0
4395,2026-03,15233.0,220.0,1.7504,1.7616,0.0134,17219.0,256.0,1.7495,231.2142,0.0
5096,2026-03,4403.0,34.0,7.5171,8.5452,0.0047,5281.0,15.0,9.2799,24.8892,1.0
2128,2026-03,37592.0,54.0,1.5625,1.6811,0.0017,102903.0,127.0,2.6339,177.2754,1.0
331,2026-03,15972.0,167.0,8.3298,8.7619,0.0105,12728.0,99.0,18.8532,107.9379,0.0
6817,2026-03,52195.0,95.0,3.8329,3.8244,0.0018,242750.0,344.0,4.5362,409.1052,0.0


rows in the 2026-03 slice: 8984


**Reading the table.** Each row is one top-10 page at one decision month, with at least 200 impressions and 20 clicks
in the two feature months. `exp_out` is what the page should earn in the next two months if its current CTR held at its
new position. `erode` compares what it actually earned against that expectation. The test origin (2026-05) and the
live origin (2026-07) have no base rate in the V1 frame receipt: the test month's outcomes are kept out of every V2 load
until the unseal step, and the live month has no outcome yet. The `lost_visibility` column is the share of eligible pages
that are unmeasurable (expected outcome clicks under 10), which matters again in ML-09.

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

**Question:** why not a fixed rule? **Why it matters (FlyRank principle: the solution hierarchy):** a model must earn its
place over a transparent rule.

- **Erosion is a combination, not one signal.** Rank, a recent CTR bump, CTR above what the position normally earns,
  demand swings and client-wide shifts all matter together (ML-06, ML-08).
- **The intuitive rules point the wrong way.** In the training months, top-3 pages eroded *less* than positions 7 to 10,
  and pages whose CTR had just risen eroded *more* than pages already slipping (ML-07).
- **No single rule holds every month.** The editor's rule scores P@50 of 0.76 in March and 0.38 on the test month,
  and the best baseline by AP changes between months (table below).

The output feeds a real content action. ML-10 maps each reason code to an editor action, with human review before any
change.

In [6]:
t = pd.DataFrame([{"month": m, "role": r, "best baseline by AP": BASE_LABEL[ev["best_baseline_by_ap"]],
                   **{f"{BASE_LABEL[b]} P@50": ev["models"][b]["p@50"] for b in BASE}} for m, r, ev in v2_months()])
display(t)

,month,role,best baseline by AP,editor's rule P@50,CTR above peers P@50,client shock P@50,persistence P@50
0,2026-02,validation,CTR above peers,0.46,0.26,0.34,0.28
1,2026-03,validation,editor's rule,0.76,0.38,0.32,0.22
2,2026-04,confirmation,editor's rule,0.50,0.30,0.56,0.38
3,2026-05,test,editor's rule,0.38,0.32,0.40,0.18


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.